<a href="https://colab.research.google.com/github/milvus-io/bootcamp/blob/master/integration/evaluation_with_deepeval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>   <a href="https://github.com/milvus-io/bootcamp/blob/master/integration/evaluation_with_deepeval.ipynb" target="_blank">
    <img src="https://img.shields.io/badge/View%20on%20GitHub-555555?style=flat&logo=github&logoColor=white" alt="GitHub Repository"/>
</a>


# Evaluation with DeepEval

This guide demonstrates how to use [DeepEval](https://docs.confident-ai.com/) to evaluate a Retrieval-Augmented Generation (RAG) pipeline built upon [Milvus](https://milvus.io/).

The RAG system combines a retrieval system with a generative model to generate new text based on a given prompt. The system first retrieves relevant documents from a corpus using Milvus, and then uses a generative model to generate new text based on the retrieved documents.

DeepEval is a framework that helps you evaluate your RAG pipelines. There are existing tools and frameworks that help you build these pipelines but evaluating it and quantifying your pipeline performance can be hard. This is where DeepEval comes in.

## Prerequisites

Before running this notebook, make sure you have the following dependencies installed:

In [ ]:
! pip install --upgrade pymilvus openai requests tqdm pandas deepeval

> If you are using Google Colab, to enable dependencies just installed, you may need to **restart the runtime** (click on the "Runtime" menu at the top of the screen, and select "Restart session" from the dropdown menu).

We will use OpenAI as the LLM in this example. You should prepare the [api key](https://platform.openai.com/docs/quickstart) `OPENAI_API_KEY` as an environment variable.

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "sk-*****************"

## Define the RAG pipeline

We will define the RAG class that use Milvus as the vector store, and OpenAI as the LLM.
The class contains the `load` method, which loads the text data into Milvus, the `retrieve` method, which retrieves the most similar text data to the given question, and the `answer` method, which answers the given question with the retrieved knowledge.

In [ ]:
import sys
import os
from typing import List
from pathlib import Path

# Add the project root to Python path to import rag_graph
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from chatbot_graphs.rag_graph import build_graph, get_retriever


class GraphRAG:
    """
    Graph RAG (Retrieval-Augmented Generation) class built upon Neo4j Knowledge Graph.
    
    This wraps the LangGraph-based RAG system for compatibility with DeepEval evaluation.
    Uses hybrid search with Reciprocal Rank Fusion for optimal retrieval quality.
    """

    def __init__(self):
        """Initialize the Graph RAG system with Neo4j backend."""
        print("[GraphRAG] Initializing Neo4j-based Graph RAG...")
        
        # Initialize the graph and retriever
        self.graph = build_graph()
        self.retriever = get_retriever()
        
        print("[GraphRAG] Graph RAG initialized successfully")
        print("[GraphRAG] Using hybrid search with RRF fusion")

    def retrieve(self, question: str, top_k: int = 5) -> List[str]:
        """
        Retrieve relevant information using hybrid search with Reciprocal Rank Fusion.
        
        This method combines three retrieval strategies:
        1. Vector similarity search (semantic matching)
        2. Text-based keyword search (lexical matching)
        3. Entity graph search (structural matching with relationships)
        
        Results are fused using RRF for optimal ranking.
        
        Args:
            question: Query question
            top_k: Maximum number of context items to retrieve
            
        Returns:
            List of retrieved context strings
        """
        print(f"\n[GraphRAG.retrieve] Hybrid search for: {question[:60]}...")
        
        # Use the hybrid search method from retriever
        result = self.retriever.hybrid_search(
            query=question,
            limit=top_k,
            enable_vector=True,
            enable_text=True,
            enable_entity=True
        )
        
        if not result or "No results found" in result:
            return [f"No relevant information found in knowledge graph for: {question}"]
        
        # Split the formatted result into individual context items
        # Each entity starts with a numbered markdown heading like "**1. EntityName**"
        contexts = []
        current_context = []
        
        for line in result.split('\n'):
            # Check if this is a new entity (starts with **N. where N is a number)
            if line.strip().startswith('**') and '. ' in line and current_context:
                # Save the previous context
                contexts.append('\n'.join(current_context))
                current_context = [line]
            else:
                current_context.append(line)
        
        # Add the last context
        if current_context:
            contexts.append('\n'.join(current_context))
        
        # Clean up empty contexts
        contexts = [ctx.strip() for ctx in contexts if ctx.strip()]
        
        if not contexts:
            return [result]  # Return as single context if splitting failed
        
        print(f"[GraphRAG.retrieve] Returning {len(contexts)} hybrid-fused contexts\n")
        return contexts[:top_k]

    def load(self, texts: List[str]):
        """
        Load text data into Neo4j knowledge graph.
        
        Note: This assumes you have already processed and loaded your data into Neo4j
        using the knowledge graph pipeline. This method is kept for interface compatibility
        but doesn't perform the actual loading.
        
        Args:
            texts: List of text documents (not used in graph RAG as data is pre-loaded)
        """
        print(f"[GraphRAG] Note: Graph RAG uses pre-loaded Neo4j knowledge graph.")
        print(f"[GraphRAG] Skipping load of {len(texts)} texts - ensure your Neo4j database is populated.")
        print(f"[GraphRAG] Use the knowledge graph pipeline to process and load documents into Neo4j.")

    def answer(
        self,
        question: str,
        retrieval_top_k: int = 5,
        return_retrieved_text: bool = False,
    ):
        """
        Answer a question using the Graph RAG system.
        
        This invokes the full LangGraph pipeline with:
        - Query preprocessing (intent detection, entity extraction)
        - ReAct agent for tool selection and reasoning
        - Hybrid retrieval with RRF fusion
        
        Args:
            question: Question to answer
            retrieval_top_k: Number of context items to retrieve (used for retrieval only)
            return_retrieved_text: Whether to return retrieved contexts along with answer
            
        Returns:
            Answer string, or tuple of (answer, contexts) if return_retrieved_text=True
        """
        # Create initial state
        initial_state = {
            "messages": [HumanMessage(content=question)],
            "retrieved_context": "",
            "intent": "",
            "original_question": "",
            "entities": []
        }
        
        # Run the graph
        print(f"\n[GraphRAG] Processing question: {question[:80]}...")
        result = self.graph.invoke(initial_state)
        
        # Extract the answer from the last AI message
        answer = None
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and not msg.tool_calls:
                answer = msg.content
                break
        
        if not answer:
            answer = "I couldn't generate an answer. Please try rephrasing your question."
        
        # If contexts are needed, retrieve them separately using hybrid search for DeepEval
        if return_retrieved_text:
            contexts = self.retrieve(question, top_k=retrieval_top_k)
            return answer, contexts
        
        return answer

    def answer_with_graph_context(
        self,
        question: str,
        return_retrieved_text: bool = False,
    ):
        """
        Answer a question and extract the actual retrieved contexts from tool calls.
        
        This method inspects the agent's tool calls to get the actual retrieved contexts
        used during reasoning, providing more accurate evaluation data.
        
        Args:
            question: Question to answer
            return_retrieved_text: Whether to return retrieved contexts
            
        Returns:
            Answer string, or tuple of (answer, contexts) if return_retrieved_text=True
        """
        # Create initial state
        initial_state = {
            "messages": [HumanMessage(content=question)],
            "retrieved_context": "",
            "intent": "",
            "original_question": "",
            "entities": []
        }
        
        # Run the graph
        print(f"\n[GraphRAG] Processing question: {question[:80]}...")
        result = self.graph.invoke(initial_state)
        
        # Extract answer and contexts from messages
        answer = None
        contexts = []
        
        for msg in result["messages"]:
            # Collect tool results (retrieved contexts)
            if isinstance(msg, ToolMessage):
                # Only include substantial tool results
                if len(msg.content) > 50 and "not found" not in msg.content.lower():
                    contexts.append(msg.content)
            
            # Get final answer
            elif isinstance(msg, AIMessage) and not msg.tool_calls:
                answer = msg.content
        
        if not answer:
            answer = "I couldn't generate an answer. Please try rephrasing your question."
        
        if not contexts:
            contexts = [f"No relevant context retrieved for: {question}"]
        
        if return_retrieved_text:
            return answer, contexts
        
        return answer

    def close(self):
        """Close the Neo4j connection."""
        if hasattr(self, 'retriever'):
            self.retriever.close()
            print("[GraphRAG] Neo4j connection closed")

Let's initialize the RAG class with OpenAI and Milvus clients.

In [ ]:
# Initialize GraphRAG
# NOTE: Ensure your Neo4j database is running and populated with knowledge graph data
# The GraphRAG system requires:
# 1. Neo4j database with populated knowledge graph (entities, relationships, embeddings)
# 2. Environment variables set in .env file (NEO4J_URI, NEO4J_PASSWORD, GOOGLE_API_KEY, etc.)
# 3. OPENAI_API_KEY for the LLM agent

my_rag = GraphRAG()

# Note: The load() method is no-op for GraphRAG
# Data should be pre-loaded into Neo4j using the knowledge graph pipeline
# Example:
# from knowledge_graphs.pipeline.batch_processor import process_documents
# process_documents(input_dir="./financebench_data", schema_file="./knowledge_graphs/schema/financebench.schema")

> As for the argument of `MilvusClient`:
> - Setting the `uri` as a local file, e.g.`./milvus.db`, is the most convenient method, as it automatically utilizes [Milvus Lite](https://milvus.io/docs/milvus_lite.md) to store all data in this file.
> - If you have large scale of data, you can set up a more performant Milvus server on [docker or kubernetes](https://milvus.io/docs/quickstart.md). In this setup, please use the server uri, e.g.`http://localhost:19530`, as your `uri`.
> - If you want to use [Zilliz Cloud](https://zilliz.com/cloud), the fully managed cloud service for Milvus, adjust the `uri` and `token`, which correspond to the [Public Endpoint and Api key](https://docs.zilliz.com/docs/on-zilliz-cloud-console#free-cluster-details) in Zilliz Cloud.

## Run the RAG pipeline and get results

We use the [Milvus development guide](https://github.com/milvus-io/milvus/blob/master/DEVELOPMENT.md) to be as the private knowledge in our RAG, which is a good data source for a simple RAG pipeline.

Download it and load it into the rag pipeline.

In [ ]:
import urllib.request
import os

url = "https://raw.githubusercontent.com/milvus-io/milvus/master/DEVELOPMENT.md"
file_path = "./Milvus_DEVELOPMENT.md"

if not os.path.exists(file_path):
    urllib.request.urlretrieve(url, file_path)
with open(file_path, "r") as file:
    file_text = file.read()

text_lines = file_text.split("# ")
my_rag.load(text_lines)

Creating embeddings: 100%|██████████| 47/47 [00:20<00:00,  2.26it/s]


Let's define a query question about the content of the development guide documentation. And then use the `answer` method to get the answer and the retrieved context texts.

In [ ]:
question = "what is the hardware requirements specification if I want to build Milvus and run from source code?"
my_rag.answer(question, return_retrieved_text=True)

('The hardware requirements specification to build and run Milvus from source code is as follows:\n\n- 8GB of RAM\n- 50GB of free disk space',
 ['Hardware Requirements\n\nThe following specification (either physical or virtual machine resources) is recommended for Milvus to build and run from source code.\n\n```\n- 8GB of RAM\n- 50GB of free disk space\n```\n\n##',
  'Building Milvus on a local OS/shell environment\n\nThe details below outline the hardware and software requirements for building on Linux and MacOS.\n\n##',
  "Software Requirements\n\nAll Linux distributions are available for Milvus development. However a majority of our contributor worked with Ubuntu or CentOS systems, with a small portion of Mac (both x86_64 and Apple Silicon) contributors. If you would like Milvus to build and run on other distributions, you are more than welcome to file an issue and contribute!\n\nHere's a list of verified OS types where Milvus can successfully build and run:\n\n- Debian/Ubuntu\n- Am

Now let's prepare some questions with its corresponding ground truth answers. We get answers and contexts from our RAG pipeline.

In [ ]:
from datasets import Dataset
import pandas as pd

question_list = [
    "what is the hardware requirements specification if I want to build Milvus and run from source code?",
    "What is the programming language used to write Knowhere?",
    "What should be ensured before running code coverage?",
]
ground_truth_list = [
    "If you want to build Milvus and run from source code, the recommended hardware requirements specification is:\n\n- 8GB of RAM\n- 50GB of free disk space.",
    "The programming language used to write Knowhere is C++.",
    "Before running code coverage, you should make sure that your code changes are covered by unit tests.",
]
contexts_list = []
answer_list = []
for question in tqdm(question_list, desc="Answering questions"):
    answer, contexts = my_rag.answer(question, return_retrieved_text=True)
    contexts_list.append(contexts)
    answer_list.append(answer)

df = pd.DataFrame(
    {
        "question": question_list,
        "contexts": contexts_list,
        "answer": answer_list,
        "ground_truth": ground_truth_list,
    }
)
rag_results = Dataset.from_pandas(df)
df

/Users/eureka/miniconda3/envs/zilliz/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Answering questions: 100%|██████████| 3/3 [00:03<00:00,  1.06s/it]


,question,contexts,answer,ground_truth
0,what is the hardware requirements specificatio...,[Hardware Requirements\n\nThe following specif...,The hardware requirements specification to bui...,If you want to build Milvus and run from sourc...
1,What is the programming language used to write...,[CMake & Conan\n\nThe algorithm library of Mil...,The programming language used to write Knowher...,The programming language used to write Knowher...
2,What should be ensured before running code cov...,[Code coverage\n\nBefore submitting your pull ...,"Before running code coverage, it should be ens...","Before running code coverage, you should make ..."


## Evaluating Retriever

When evaluating a retriever in large language model (LLM) systems, it's crucial to assess the following:

1. **Ranking Relevance**: How effectively the retriever prioritizes relevant information over irrelevant data.
   
2. **Contextual Retrieval**: The ability to capture and retrieve contextually relevant information based on the input.

3. **Balance**: How well the retriever manages text chunk size and retrieval scope to minimize irrelevancies.

Together, these factors provide a comprehensive understanding of how the retriever prioritizes, captures, and presents the most useful information.

In [ ]:
from deepeval.metrics import (
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
)
from deepeval.test_case import LLMTestCase
from deepeval import evaluate

contextual_precision = ContextualPrecisionMetric()
contextual_recall = ContextualRecallMetric()
contextual_relevancy = ContextualRelevancyMetric()

test_cases = []

for index, row in df.iterrows():
    test_case = LLMTestCase(
        input=row["question"],
        actual_output=row["answer"],
        expected_output=row["ground_truth"],
        retrieval_context=row["contexts"],
    )
    test_cases.append(test_case)

# test_cases
result = evaluate(
    test_cases=test_cases,
    metrics=[contextual_precision, contextual_recall, contextual_relevancy],
    print_results=False,  # Change to True to see detailed metric results
)

/Users/eureka/miniconda3/envs/zilliz/lib/python3.9/site-packages/deepeval/__init__.py:49: UserWarning: You are using deepeval version 1.1.6, however version 1.2.2 is available. You should consider upgrading via the "pip install --upgrade deepeval" command.
  warnings.warn(


✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4o, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4o, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4o, strict=False, async_mode=True)...

Event loop is already running. Applying nest_asyncio patch to allow async execution...


Evaluating 3 test case(s) in parallel: |██████████|100% (3/3) [Time Taken: 00:11,  3.91s/test case]


✓ Tests finished 🎉! Run 'deepeval login' to view evaluation results on Confident AI. 
‼️  NOTE: You can also run evaluations on ALL of deepeval's metrics directly on Confident AI instead.

## Evaluating Generation

To assess the quality of generated outputs in large language models (LLMs), it's important to focus on two key aspects:

1. **Relevance**: Evaluate whether the prompt effectively guides the LLM to generate helpful and contextually appropriate responses.
   
2. **Faithfulness**: Measure the accuracy of the output, ensuring the model produces information that is factually correct and free from hallucinations or contradictions. The generated content should align with the factual information provided in the retrieval context.

These factors together ensure that the outputs are both relevant and reliable.

In [ ]:
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase
from deepeval import evaluate

answer_relevancy = AnswerRelevancyMetric()
faithfulness = FaithfulnessMetric()

test_cases = []

for index, row in df.iterrows():
    test_case = LLMTestCase(
        input=row["question"],
        actual_output=row["answer"],
        expected_output=row["ground_truth"],
        retrieval_context=row["contexts"],
    )
    test_cases.append(test_case)

# test_cases
result = evaluate(
    test_cases=test_cases,
    metrics=[answer_relevancy, faithfulness],
    print_results=False,  # Change to True to see detailed metric results
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o, strict=False, async_mode=True)...

Event loop is already running. Applying nest_asyncio patch to allow async execution...


Evaluating 3 test case(s) in parallel: |██████████|100% (3/3) [Time Taken: 00:11,  3.97s/test case]


✓ Tests finished 🎉! Run 'deepeval login' to view evaluation results on Confident AI. 
‼️  NOTE: You can also run evaluations on ALL of deepeval's metrics directly on Confident AI instead.